# Black-Litterman portfolio allocation for a net-zero transition

## What this notebook is trying to solve

The ETHack bonus question asks:

> If the world committed tomorrow to reaching net zero as fast as possible, how should a $1bn S&P 500 portfolio be allocated?

This notebook treats that as a portfolio-construction problem, not just a sustainability-ranking problem. The existing Climate Transition Score measures which companies look more prepared for a rapid emissions transition. The portfolio still needs to account for historical return risk, correlations, benchmark information and diversification constraints.

The core idea is therefore:

$$
\text{Net-zero allocation} = \text{market prior} + \text{transition-preparedness views} + \text{risk constraints}
$$

The final output is a table of optimized portfolio weights and dollar allocations for a $1bn long-only S&P 500 portfolio.


## Basic idea

A direct score-weighted portfolio would put the largest weights on the highest Climate Transition Scores. That is easy to understand, but it is not a complete investment model because it ignores three important facts:

1. High-scoring companies can still be risky or highly correlated with each other.
2. A low-scoring company can still belong in a diversified portfolio at a small weight.
3. The sustainability score is uncertain and should not be treated as a perfect return forecast.

Black-Litterman is useful because it starts with market-implied expected returns and then applies explicit, auditable views. In this notebook, the view is that companies with stronger transition preparedness should outperform companies with weaker preparedness if the world suddenly accelerates toward net zero.


## Black-Litterman intuition and notation

The Black-Litterman prior is the vector of equilibrium excess returns implied by the market portfolio:

$$
\Pi = \delta \Sigma w_{mkt}
$$

where:

- $\Sigma$ is the annualized covariance matrix of stock returns.
- $w_{mkt}$ is the benchmark market portfolio.
- $\delta$ is the investor's risk-aversion parameter.
- $\Pi$ is the prior expected-return vector implied by market equilibrium.

The model then combines $\Pi$ with investor views:

- $P$ maps each view to assets. A row of $P$ is a long-short portfolio.
- $Q$ is the expected return for each view.
- $\Omega$ is the uncertainty of each view.
- $\tau$ controls uncertainty in the prior.
- $\mu_{BL}$ is the posterior expected-return vector after blending the prior and the views.

In this notebook, the Climate Transition Score is interpreted as **transition preparedness**. It enters through $P$, $Q$ and $\Omega$, not as a direct portfolio weight. This preserves the distinction between a sustainability signal and an investable portfolio.


## Pipeline

The notebook follows the same transparent logic as the Climate Transition Score notebook:

1. Load the already available project data.
2. Align the common S&P 500 ticker universe.
3. Compute daily returns and an annualized covariance matrix.
4. Build the market-equilibrium prior \(\Pi=\delta\Sigma w_{mkt}\).
5. Convert the Climate Transition Score into a relative Black-Litterman view.
6. Set view uncertainty \(\Omega\) using data coverage as a confidence proxy.
7. Compute posterior expected returns \(\mu_{BL}\).
8. Solve a constrained mean-variance portfolio.
9. Report weights, $1bn dollar allocations and diagnostics.


## Assumptions

The portfolio result depends on explicit assumptions. They are stated here so the notebook can be challenged or modified without hunting through the code.

- **Prices:** local adjusted daily prices from `data/sp500_10yr_prices.csv`.
- **Sustainability:** local Climate Transition Score from `outputs/climate_transition_scores.csv`.
- **Returns:** daily simple returns.
- **Annualization:** 252 trading days.
- **Covariance:** annualized sample covariance of daily returns.
- **Benchmark prior:** market-cap weights if `data/sp500_market_caps.csv` exists; otherwise an equal-weight S&P 500 proxy.
- **Risk aversion:** $\delta=2.5$, a moderate standard choice for annual expected-return units.
- **Prior uncertainty:** $\tau=0.05$.
- **Transition view:** top-quintile transition-prepared firms outperform bottom-quintile firms by 3% per year.
- **View confidence:** average data coverage of the view companies, clipped between 25% and 85%.
- **Portfolio constraints:** long-only, fully invested, maximum 5% issuer weight.

The 3% view is not estimated from history. It is a scenario assumption: a rapid global net-zero commitment should create a relative return advantage for companies that are already better prepared for transition costs, regulation, customer shifts and stranded-asset risk.


## 1. Load libraries and define file paths

The notebook uses standard scientific Python tools. PyPortfolioOpt is used when available because it provides established portfolio routines. A short closed-form Black-Litterman and SciPy optimization fallback is kept so the notebook remains reproducible in a minimal environment.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from pypfopt import EfficientFrontier
    from pypfopt.black_litterman import BlackLittermanModel
except Exception:
    EfficientFrontier = None
    BlackLittermanModel = None

try:
    from scipy.optimize import minimize
except Exception:
    minimize = None

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

# Make the notebook robust whether it is run from the repository root or from notebooks/.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "sp500_10yr_prices.csv").exists()
)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

PRICE_PATH = DATA_DIR / "sp500_10yr_prices.csv"
SCORE_PATH = OUTPUT_DIR / "climate_transition_scores.csv"
CAP_PATH = DATA_DIR / "sp500_market_caps.csv"
ALLOCATION_OUTPUT_PATH = OUTPUT_DIR / "black_litterman_net_zero_allocations.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Price file exists: {PRICE_PATH.exists()} -> {PRICE_PATH}")
print(f"Score file exists: {SCORE_PATH.exists()} -> {SCORE_PATH}")
print(f"Market-cap file exists: {CAP_PATH.exists()} -> {CAP_PATH}")


## 2. Load and inspect the data

This section loads the two portfolio inputs and checks their dimensions before any modeling is done. The price file is already present in the project, so no external market-data download is required.


In [ ]:
prices_raw = pd.read_csv(PRICE_PATH, parse_dates=["Date"])
scores_raw = pd.read_csv(SCORE_PATH)

data_inputs = pd.DataFrame([
    {"Dataset": "Adjusted stock prices", "Path": str(PRICE_PATH.relative_to(PROJECT_ROOT)), "Rows": len(prices_raw), "Columns": prices_raw.shape[1]},
    {"Dataset": "Climate Transition Scores", "Path": str(SCORE_PATH.relative_to(PROJECT_ROOT)), "Rows": len(scores_raw), "Columns": scores_raw.shape[1]},
])

data_inputs


## 3. Align the common ticker universe

The sustainability file uses Bloomberg-style identifiers such as `AAPL UW Equity`. The price file uses clean stock tickers such as `AAPL`. The join key is therefore the first token of the Bloomberg identifier.

The model keeps only companies with both a transition score and enough price history for return estimation. This prevents the optimizer from allocating to names whose risk cannot be estimated reliably.


In [ ]:
prices = prices_raw.set_index("Date").sort_index()
prices.columns = prices.columns.astype(str).str.strip().str.replace("/", "-", regex=False)
prices = prices.apply(pd.to_numeric, errors="coerce")

scores = scores_raw.copy()
scores["Ticker"] = scores["ID"].astype(str).str.split().str[0].str.replace("/", "-", regex=False)
scores = (
    scores.dropna(subset=["Ticker", "Climate_Transition_Score"])
    .drop_duplicates("Ticker")
    .set_index("Ticker")
)

common_tickers = prices.columns.intersection(scores.index).sort_values()
prices = prices[common_tickers]
scores = scores.loc[common_tickers]

# Require broad price coverage, then forward-fill short gaps in adjusted prices.
min_price_coverage = 0.80
prices = prices.dropna(axis=1, thresh=int(min_price_coverage * len(prices))).ffill().dropna()
scores = scores.loc[prices.columns]

universe_summary = pd.Series({
    "Common tickers before price-history filter": len(common_tickers),
    "Tickers after price-history filter": len(prices.columns),
    "First price date": prices.index.min().date(),
    "Last price date": prices.index.max().date(),
})

universe_summary


## 4. Compute returns and covariance

The optimizer needs a risk model. This notebook uses daily simple returns and an annualized sample covariance matrix:

$$
\Sigma = 252 \times \operatorname{Cov}(r_{daily})
$$

A sample covariance estimator is transparent and easy to audit. More advanced estimators, such as Ledoit-Wolf shrinkage, would be reasonable extensions but are not necessary for the baseline notebook.


In [ ]:
TRADING_DAYS = 252

returns = prices.pct_change().dropna(how="all")
returns = returns.dropna(axis=1, thresh=int(0.95 * len(returns))).dropna()
Sigma = returns.cov() * TRADING_DAYS
scores = scores.loc[returns.columns]

risk_summary = pd.Series({
    "Return observations": len(returns),
    "Assets in covariance matrix": Sigma.shape[0],
    "Average annualized volatility": np.sqrt(np.diag(Sigma)).mean(),
    "Median annualized volatility": np.median(np.sqrt(np.diag(Sigma))),
})

risk_summary


## 5. Build the market-equilibrium prior

Black-Litterman starts from the return vector implied by the benchmark portfolio:

$$
\Pi = \delta \Sigma w_{mkt}
$$

A true S&P 500 prior should use market-cap weights. If a local market-cap file is not available, the notebook uses an equal-weight proxy and labels it clearly. This is preferable to inventing market capitalizations from price levels, because price alone does not identify company size.


In [ ]:
risk_aversion_delta = 2.5

default_weight = 1 / Sigma.shape[0]

if CAP_PATH.exists():
    caps = pd.read_csv(CAP_PATH)
    caps.columns = caps.columns.astype(str).str.strip()
    ticker_col = next(c for c in caps.columns if c.lower() in {"ticker", "symbol", "id"})
    cap_col = next(c for c in caps.columns if c.lower() in {"market_cap", "market cap", "mkt_cap", "capitalization"})
    caps[ticker_col] = caps[ticker_col].astype(str).str.split().str[0].str.replace("/", "-", regex=False)
    market_caps = caps.set_index(ticker_col)[cap_col].astype(float).reindex(Sigma.index)
    market_caps = market_caps.fillna(market_caps.dropna().mean())
    w_mkt = market_caps / market_caps.sum()
    benchmark_label = "market-cap weighted proxy"
else:
    w_mkt = pd.Series(default_weight, index=Sigma.index, name="Benchmark_Weight")
    benchmark_label = "equal-weight proxy"

pi = pd.Series(risk_aversion_delta * Sigma.values @ w_mkt.values, index=Sigma.index, name="Prior_Return")

prior_summary = pd.Series({
    "Benchmark used": benchmark_label,
    "Risk aversion delta": risk_aversion_delta,
    "Minimum benchmark weight": w_mkt.min(),
    "Maximum benchmark weight": w_mkt.max(),
    "Average prior return": pi.mean(),
})

prior_summary


## 6. Convert transition preparedness into BL views

The Climate Transition Score is strongest as a cross-sectional ranking. The baseline view is therefore relative rather than absolute:

$$
\text{Top transition-prepared quintile} - \text{Bottom transition-prepared quintile} = 3\%
$$

This says that, under a rapid net-zero scenario, companies already prepared for emissions reduction should earn a positive annual return premium over companies that look least prepared. The view is expressed as a long-short portfolio row in \(P\): equal-weight long the top quintile and equal-weight short the bottom quintile.


In [ ]:
score = scores["Climate_Transition_Score"].astype(float)
high = score[score >= score.quantile(0.80)].index
low = score[score <= score.quantile(0.20)].index

P = pd.DataFrame(0.0, index=["High minus low transition preparedness"], columns=Sigma.columns)
P.loc[:, high] = 1 / len(high)
P.loc[:, low] = -1 / len(low)

Q = pd.Series([0.03], index=P.index, name="View_Return")

view_composition = pd.DataFrame({
    "Group": ["High transition preparedness", "Low transition preparedness"],
    "Stocks": [len(high), len(low)],
    "Average score": [score.loc[high].mean(), score.loc[low].mean()],
    "Average data coverage": [scores.loc[high, "Data_Coverage"].astype(float).mean(), scores.loc[low, "Data_Coverage"].astype(float).mean()],
})

view_composition


## 7. Define view uncertainty $\Omega$

The view uncertainty should be smaller when the view is more reliable and larger when the view is noisy. This notebook uses two ingredients:

1. The historical variance of the long-short view portfolio, $P\Sigma P^\top$.
2. The average data coverage of the companies used in the view.

The confidence value is clipped between 25% and 85%. The cap avoids treating ESG data as perfect even when coverage is high; the floor avoids discarding the view completely when coverage is imperfect.


In [ ]:
tau = 0.05
view_variance = float((P.values @ Sigma.values @ P.values.T)[0, 0])
avg_coverage = scores.loc[high.union(low), "Data_Coverage"].astype(float).mean()
view_confidence = float(np.clip(avg_coverage, 0.25, 0.85))

Omega = pd.DataFrame(
    [[view_variance * (1 - view_confidence) / view_confidence]],
    index=P.index,
    columns=P.index,
)

view_parameters = pd.DataFrame({
    "View": P.index,
    "Q annual return": Q.values,
    "Long-short variance": view_variance,
    "Confidence": view_confidence,
    "Omega": np.diag(Omega),
    "Tau": tau,
})

view_parameters


## 8. Compute Black-Litterman posterior returns

The posterior return vector $\mu_{BL}$ blends the market prior with the transition view. Companies in the high-score group should generally receive positive posterior revisions, while companies in the low-score group should generally receive negative revisions. The size of the revision depends on covariance and view confidence, not just the raw score.


In [ ]:
if BlackLittermanModel is not None:
    bl = BlackLittermanModel(
        Sigma,
        pi=pi,
        P=P,
        Q=Q,
        omega=Omega,
        tau=tau,
        risk_aversion=risk_aversion_delta,
    )
    mu_bl = bl.bl_returns().rename("BL_Posterior_Return")
else:
    # Standard BL posterior mean; all inputs are annualized to match Sigma and Q.
    tau_Sigma_inv = np.linalg.inv(tau * Sigma.values)
    Omega_inv = np.linalg.inv(Omega.values)
    posterior_precision = tau_Sigma_inv + P.values.T @ Omega_inv @ P.values
    posterior_mean_term = tau_Sigma_inv @ pi.values + P.values.T @ Omega_inv @ Q.values
    mu_bl = pd.Series(
        np.linalg.solve(posterior_precision, posterior_mean_term),
        index=Sigma.index,
        name="BL_Posterior_Return",
    )

posterior = pd.DataFrame({
    "Company": scores["Company"],
    "Sector": scores["GICS Sector"],
    "Climate_Transition_Score": score,
    "Prior_Return": pi,
    "BL_Posterior_Return": mu_bl,
    "Posterior_Minus_Prior": mu_bl - pi,
}).sort_values("Posterior_Minus_Prior", ascending=False)

posterior.head(10)


## 9. Optimize the constrained portfolio

The final allocation solves a long-only quadratic-utility problem:

$$
\max_w \; \mu_{BL}^{\top}w - \frac{\delta}{2} w^{\top}\Sigma w
$$

subject to:

$$
\sum_i w_i = 1, \quad 0 \leq w_i \leq 5\%
$$

The 5% issuer cap prevents the optimizer from turning one or two attractive posterior returns into an unrealistic concentrated fund. The result is a diversified transition-aware portfolio, not a pure list of climate leaders.


In [ ]:
portfolio_value = 1_000_000_000
max_weight = 0.05

if EfficientFrontier is not None:
    ef = EfficientFrontier(mu_bl, Sigma, weight_bounds=(0, max_weight))
    ef.max_quadratic_utility(risk_aversion=risk_aversion_delta)
    weights = pd.Series(ef.clean_weights(), name="Weight").reindex(mu_bl.index).fillna(0.0)
else:
    if minimize is None:
        raise ImportError("Install PyPortfolioOpt or scipy to run the constrained optimization.")
    x0 = np.repeat(1 / len(mu_bl), len(mu_bl))
    bounds = [(0, max_weight)] * len(mu_bl)
    constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}
    result = minimize(
        lambda w: -(mu_bl.values @ w - 0.5 * risk_aversion_delta * w @ Sigma.values @ w),
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 1_000, "ftol": 1e-10},
    )
    if not result.success:
        raise RuntimeError(result.message)
    weights = pd.Series(result.x, index=mu_bl.index, name="Weight")

weights = weights / weights.sum()
allocations = (
    pd.DataFrame({
        "Company": scores["Company"],
        "Sector": scores["GICS Sector"],
        "Climate_Transition_Score": score,
        "Data_Coverage": scores["Data_Coverage"],
        "Benchmark_Weight": w_mkt,
        "Prior_Return": pi,
        "BL_Posterior_Return": mu_bl,
        "Weight": weights,
    })
    .assign(
        Active_Weight=lambda df: df["Weight"] - df["Benchmark_Weight"],
        Dollar_Allocation=lambda df: df["Weight"] * portfolio_value,
    )
    .sort_values("Weight", ascending=False)
)

allocations.head(25)


## 10. Sanity checks

These checks tie the numerical output back to the investment story. A reasonable result should be fully invested, long-only, within the issuer cap, tilted toward stronger transition preparedness, and still diversified across sectors.


In [ ]:
portfolio_checks = pd.Series({
    "Total weight": allocations["Weight"].sum(),
    "Total dollar allocation": allocations["Dollar_Allocation"].sum(),
    "Largest issuer weight": allocations["Weight"].max(),
    "Number of non-zero positions": (allocations["Weight"] > 1e-6).sum(),
    "Weighted average transition score": np.average(allocations["Climate_Transition_Score"], weights=allocations["Weight"]),
    "Benchmark average transition score": np.average(score.reindex(w_mkt.index), weights=w_mkt),
})

portfolio_checks


## 11. Diagnostics

The diagnostics are deliberately compact:

- prior versus posterior returns shows how much the BL view changed expected returns;
- score versus weight shows whether the portfolio tilt is directionally sensible;
- sector exposure shows whether the transition view creates large sector concentrations;
- largest positions show the actual $1bn allocation.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(posterior["Prior_Return"], posterior["BL_Posterior_Return"], alpha=0.65)
ax.axline((0, 0), slope=1, color="black", linewidth=1, linestyle="--")
ax.set_title("Prior vs Black-Litterman posterior returns")
ax.set_xlabel("Market-implied prior return")
ax.set_ylabel("BL posterior return")
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(allocations["Climate_Transition_Score"], allocations["Weight"], alpha=0.65)
ax.set_title("Transition preparedness vs optimized weight")
ax.set_xlabel("Climate Transition Score")
ax.set_ylabel("Portfolio weight")
plt.show()

sector_exposure = allocations.groupby("Sector", dropna=False)["Weight"].sum().sort_values(ascending=False)
sector_exposure.plot(kind="bar", figsize=(8, 4), title="Optimized sector exposure")
plt.ylabel("Portfolio weight")
plt.tight_layout()
plt.show()


In [ ]:
largest_positions = allocations[[
    "Company", "Sector", "Climate_Transition_Score", "Benchmark_Weight",
    "Active_Weight", "Weight", "Dollar_Allocation"
]].head(25)

largest_positions


## 12. Export portfolio output

The final allocation is saved so it can be used directly in a presentation or compared with alternative assumptions. The export includes benchmark weight, active weight, BL posterior return, transition score and dollar allocation.


In [ ]:
allocations.to_csv(ALLOCATION_OUTPUT_PATH)
print(f"Saved allocation output: {ALLOCATION_OUTPUT_PATH}")
print(f"Rows exported: {len(allocations)}")


## Final answer and interpretation

The answer is not “buy only the greenest companies.” Under the modeled rapid net-zero scenario, the $1bn portfolio should tilt toward companies with high transition preparedness, but only after accounting for covariance, benchmark information and concentration limits.

In this implementation, the Climate Transition Score creates a relative Black-Litterman view: high-preparedness companies are expected to outperform low-preparedness companies by 3% per year. Black-Litterman converts that scenario view into posterior expected returns. The constrained optimizer then allocates capital to the names with the best combination of transition preparedness, risk contribution and diversification benefit.

Main strengths:

- Uses the existing Climate Transition Score rather than creating a separate sustainability metric.
- Uses local historical stock prices already available in the project.
- Separates sustainability judgement from portfolio construction.
- Makes $P$, $Q$, $\Omega$, $\tau$, $\Pi$ and $\mu_{BL}$ explicit.
- Produces investable weights and dollar allocations for a $1bn fund.

Main limitations:

- The 3% transition premium is a scenario assumption, not a historical estimate.
- The equal-weight benchmark fallback is not a true S&P 500 cap-weighted market portfolio.
- Sample covariance can be noisy for large equity universes.
- The sustainability score measures corporate transition preparedness, not direct transition beta or stranded-asset exposure.
- Sector neutrality is not imposed; sector exposure should therefore be reviewed in the diagnostics.

Possible extensions include using true S&P 500 market-cap weights, sector-relative BL views, covariance shrinkage, tracking-error limits, transaction costs, turnover constraints and multiple transition scenarios with different values of $Q$ and $\Omega$.
